This notebook identifies and compares protein-binding interfaces from three-dimensional complex structures. Starting from an uploaded PDB file containing two interacting protein chains, it demonstrates atom- and residue-distance calculations and identifies contacting residue pairs using a minimum interatomic distance of less than 3.0 Å. A dedicated export cell saves the interface residues from both chains to a downloadable CSV file. An optional second workflow reads a separately prepared partner–residue table, FGFR1_InterfaceOverlap (3).dat, and determines whether different binding partners share contact residues on a common target protein. Shared residues indicate interface overlap and potential competition, but do not establish competitive binding. The notebook is designed for Google Colab and includes instructions for selecting chains, adjusting distance cutoffs, and preparing input files.

In [ ]:
# Install Biopython for reading and analyzing PDB structures.
# Run in Google Colab; execute this cell once at the start of a new runtime.
# Input: none. Internet access is needed for package installation.
# Output: Biopython becomes available to subsequent cells.
# Workflow: Cells 2-8 inspect distances; Cell 10 exports interface residues;
# Cells 11-14 optionally compare partner interfaces using a separate table.

!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 15.4 MB/s eta 0:00:00


In [ ]:
# Upload and inspect a protein-complex PDB file.
# Requires: Biopython from Cell 1 and Google Colab's upload interface.
# Upload one .pdb file containing both interacting proteins as separate chains
# in their complex geometry; separate unaligned monomer files are insufficient.
# Only the first uploaded filename is parsed. mmCIF is not supported here.
# Output: structure and pdb_file, plus model, chain, residue, and atom details.
# Check the chain IDs in the output before editing the A/B settings below.
# Use a single-model PDB for Cells 3-8: their loops do not separate models.

# Import necessary modules
from Bio import PDB
from google.colab import files

# Upload a PDB file
uploaded = files.upload()

# Get the file name of the uploaded PDB file
pdb_file = list(uploaded.keys())[0]

# Initialize the PDB parser
parser = PDB.PDBParser(QUIET=True)

# Parse the PDB file
structure = parser.get_structure('protein', pdb_file)

# Print the structure summary (you can customize this to print specific details)
print(f"Structure ID: {structure.get_id()}")
print(f"Number of models: {len(structure)}")

# Loop through the chains and residues
for model in structure:
    for chain in model:
        print(f"Chain {chain.get_id()}:")
        for residue in chain:
            print(f"  Residue: {residue.get_resname()} {residue.get_id()[1]}")
            for atom in residue:
                # Extract the atom name and its coordinates (x, y, z)
                atom_name = atom.get_name()
                coordinates = atom.get_coord()  # Returns a numpy array [x, y, z]

                # Print atom name and its coordinates
                print(f"Atom: {atom_name}, Coordinates: {coordinates}")

Streaming output truncated to the last 5000 lines.
Atom: C, Coordinates: [-6.023 18.452 13.472]
Atom: O, Coordinates: [-4.889 18.363 13.942]
Atom: CB, Coordinates: [-6.076 20.875 12.694]
Atom: CG, Coordinates: [-5.808 20.495 11.221]
Atom: CD1, Coordinates: [-7.084 20.264 10.432]
Atom: CD2, Coordinates: [-5.014 21.609 10.544]
  Residue: HIS 145
Atom: N, Coordinates: [-6.632 17.45  12.855]
Atom: CA, Coordinates: [-6.043 16.149 12.606]
Atom: C, Coordinates: [-5.96  15.918 11.1  ]
Atom: O, Coordinates: [-6.975 15.671 10.451]
Atom: CB, Coordinates: [-6.856 15.067 13.329]
Atom: CG, Coordinates: [-6.732 15.123 14.828]
Atom: ND1, Coordinates: [-6.129 14.176 15.633]
Atom: CD2, Coordinates: [-7.166 16.13  15.664]
Atom: CE1, Coordinates: [-6.211 14.587 16.905]
Atom: NE2, Coordinates: [-6.829 15.784 16.957]
  Residue: ALA 146
Atom: N, Coordinates: [-4.756 16.006 10.556]
Atom: CA, Coordinates: [-4.467 15.539  9.205]
Atom: C, Coordinates: [-4.073 14.057  9.292]
Atom: O, Coordinates: [-3.104 13.718  

In [ ]:
# Demonstrate distances from one atom to the other protein.
# Requires: structure from Cell 2. Select both chain IDs below (default A/B).
# protein1_atoms[1] selects the SECOND atom, so that chain needs >=2 atoms.
# Output: distances in Angstrom from that atom to every atom in the other
# chain, with only the first 10 printed. No interface cutoff is applied.
# This demonstration is not a list of interface residues.

import math  # Import the math module

def get_atoms_coordinates_from_chain(protein_structure, chain_id):
    atom_coordinates = []
    for model in protein_structure:
        for chain in model:
            if chain.get_id() == chain_id:  # Filter atoms by chain ID
                for residue in chain:
                    for atom in residue:
                        atom_coordinates.append(atom.get_coord())  # Get atom coordinates
    return atom_coordinates

# Example: Choose chain 'A' as protein 1 and chain 'B' as protein 2
chain_id_protein1 = 'A'  # Protein 1 chain
chain_id_protein2 = 'B'  # Protein 2 chain

# Get atom coordinates for both chains (proteins)
protein1_atoms = get_atoms_coordinates_from_chain(structure, chain_id_protein1)
protein2_atoms = get_atoms_coordinates_from_chain(structure, chain_id_protein2)

# Function to calculate Euclidean distance between two points
def calculate_distance(atom1, atom2):
    return math.sqrt((atom2[0] - atom1[0])**2 + (atom2[1] - atom1[1])**2 + (atom2[2] - atom1[2])**2)

# Choose an atom in protein 1 (for example, the first atom in protein 1)
atom_in_protein1 = protein1_atoms[1]

# Calculate the distance from this atom in protein 1 to all atoms in protein 2
distances = []
for atom_in_protein2 in protein2_atoms:
    distance = calculate_distance(atom_in_protein1, atom_in_protein2)
    distances.append(distance)

# Print the distances (for simplicity, print the first 10 distances)
for i, distance in enumerate(distances[:10]):
    print(f"Distance from atom {i+1} in protein 2 to the chosen atom in protein 1: {distance:.3f} Å")

Distance from atom 1 in protein 2 to the chosen atom in protein 1: 55.813 Å
Distance from atom 2 in protein 2 to the chosen atom in protein 1: 56.675 Å
Distance from atom 3 in protein 2 to the chosen atom in protein 1: 56.191 Å
Distance from atom 4 in protein 2 to the chosen atom in protein 1: 54.989 Å
Distance from atom 5 in protein 2 to the chosen atom in protein 1: 56.659 Å
Distance from atom 6 in protein 2 to the chosen atom in protein 1: 57.498 Å
Distance from atom 7 in protein 2 to the chosen atom in protein 1: 57.522 Å
Distance from atom 8 in protein 2 to the chosen atom in protein 1: 58.406 Å
Distance from atom 9 in protein 2 to the chosen atom in protein 1: 58.655 Å
Distance from atom 10 in protein 2 to the chosen atom in protein 1: 58.086 Å


In [ ]:
import math  # Import the math module

def get_atoms_coordinates_from_chain(protein_structure, chain_id):
    atom_coordinates = []
    for model in protein_structure:
        for chain in model:
            if chain.get_id() == chain_id:  # Filter atoms by chain ID
                for residue in chain:
                    for atom in residue:
                        atom_coordinates.append(atom.get_coord())  # Get atom coordinates
    return atom_coordinates

# Example: Protein 1 is in chain 'A' and Protein 2 is in chain 'B'
chain_id_protein1 = 'A'  # Protein 1 chain
chain_id_protein2 = 'B'  # Protein 2 chain

# Get atom coordinates for both proteins (chains A and B)
protein1_atoms = get_atoms_coordinates_from_chain(structure, chain_id_protein1)
protein2_atoms = get_atoms_coordinates_from_chain(structure, chain_id_protein2)

# Function to calculate Euclidean distance between two points
def calculate_distance(atom1, atom2):
    return math.sqrt((atom2[0] - atom1[0])**2 + (atom2[1] - atom1[1])**2 + (atom2[2] - atom1[2])**2)

# Calculate the distance from each atom in protein 1 to each atom in protein 2
distances = []
for atom_in_protein1 in protein1_atoms:
    for atom_in_protein2 in protein2_atoms:
        distance = calculate_distance(atom_in_protein1, atom_in_protein2)
        distances.append(distance)

# Print the distances (for simplicity, print the first 10 distances)
for i, distance in enumerate(distances[:10]):
    print(f"Distance {i+1}: {distance:.3f} Å")

Distance 1: 54.944 Å
Distance 2: 55.822 Å
Distance 3: 55.369 Å
Distance 4: 54.172 Å
Distance 5: 55.794 Å
Distance 6: 56.606 Å
Distance 7: 56.619 Å
Distance 8: 57.479 Å
Distance 9: 57.716 Å
Distance 10: 57.153 Å


In [ ]:
import math  # Import the math module

def get_atoms_coordinates_from_residue(protein_structure, chain_id, residue_id):
    atom_coordinates = []
    for model in protein_structure:
        for chain in model:
            if chain.get_id() == chain_id:  # Filter atoms by chain ID
                for residue in chain:
                    if residue.get_id()[1] == residue_id:  # Filter by residue ID
                        for atom in residue:
                            atom_coordinates.append(atom.get_coord())  # Get atom coordinates
    return atom_coordinates

# Example: Calculate the distance between residue 25 of chain 'A' and residue 50 of chain 'B'
chain_id_protein1 = 'A'  # Protein 1 chain
chain_id_protein2 = 'B'  # Protein 2 chain
residue_id_protein1 = 25  # Residue ID of protein 1 (chain A)
residue_id_protein2 = 50  # Residue ID of protein 2 (chain B)

# Get atom coordinates for both residues
residue1_atoms = get_atoms_coordinates_from_residue(structure, chain_id_protein1, residue_id_protein1)
residue2_atoms = get_atoms_coordinates_from_residue(structure, chain_id_protein2, residue_id_protein2)

# Function to calculate Euclidean distance between two points
def calculate_distance(atom1, atom2):
    return math.sqrt((atom2[0] - atom1[0])**2 + (atom2[1] - atom1[1])**2 + (atom2[2] - atom1[2])**2)

# Calculate the distance from each atom in residue 1 to each atom in residue 2
distances = []
for atom_in_residue1 in residue1_atoms:
    for atom_in_residue2 in residue2_atoms:
        distance = calculate_distance(atom_in_residue1, atom_in_residue2)
        distances.append(distance)

# Print the distances (for simplicity, print the first 10 distances)
for i, distance in enumerate(distances[:10]):
    print(f"Distance {i+1}: {distance:.3f} Å")

Distance 1: 46.173 Å
Distance 2: 47.099 Å
Distance 3: 48.192 Å
Distance 4: 47.949 Å
Distance 5: 46.393 Å
Distance 6: 47.405 Å
Distance 7: 45.309 Å
Distance 8: 46.214 Å
Distance 9: 47.164 Å
Distance 10: 48.229 Å


In [ ]:
# Compare residues using their C-alpha (CA) coordinates.
# Requires: structure from Cell 2 and math imported in an earlier cell.
# Select both chain IDs below (default A/B).
# Output: residue_distances contains (residue_A, residue_B, CA distance);
# only the first 10 pairs are printed, with distances in Angstrom.
# Residues without CA are skipped. These CA distances differ from the
# minimum all-atom distances used to identify contacts in Cells 7-8.
# Use a single model and unique integer residue numbers.

def get_Calpha_coordinates_from_chain(protein_structure, chain_id):
    residue_coordinates = {}
    for model in protein_structure:
        for chain in model:
            if chain.get_id() == chain_id:
                for residue in chain:
                    if residue.has_id('CA'):  # Only consider residues with a Cα atom
                        ca_atom = residue['CA']
                        residue_coordinates[residue.get_id()[1]] = ca_atom.get_coord()  # Store coordinates with residue ID
    return residue_coordinates

# Example: Protein 1 is in chain 'A' and Protein 2 is in chain 'B'
chain_id_protein1 = 'A'  # Protein 1 chain
chain_id_protein2 = 'B'  # Protein 2 chain

# Get Cα atom coordinates for both proteins (chains A and B)
protein1_residues = get_Calpha_coordinates_from_chain(structure, chain_id_protein1)
protein2_residues = get_Calpha_coordinates_from_chain(structure, chain_id_protein2)

# Function to calculate Euclidean distance between two points
def calculate_distance(atom1, atom2):
    return math.sqrt((atom2[0] - atom1[0])**2 + (atom2[1] - atom1[1])**2 + (atom2[2] - atom1[2])**2)

# Calculate residue-based distance between all residues in protein 1 and all residues in protein 2
residue_distances = []

for res_id_1, ca_coord_1 in protein1_residues.items():
    for res_id_2, ca_coord_2 in protein2_residues.items():
        distance = calculate_distance(ca_coord_1, ca_coord_2)
        residue_distances.append((res_id_1, res_id_2, distance))

# Print the distances (for simplicity, print the first 10 distances)
for i, (res_id_1, res_id_2, distance) in enumerate(residue_distances[:10]):
    print(f"Distance between residue {res_id_1} (chain {chain_id_protein1}) and residue {res_id_2} (chain {chain_id_protein2}): {distance:.3f} Å")

Distance between residue 1 (chain A) and residue 1 (chain B): 56.675 Å
Distance between residue 1 (chain A) and residue 2 (chain B): 56.646 Å
Distance between residue 1 (chain A) and residue 3 (chain B): 54.428 Å
Distance between residue 1 (chain A) and residue 4 (chain B): 56.971 Å
Distance between residue 1 (chain A) and residue 5 (chain B): 57.435 Å
Distance between residue 1 (chain A) and residue 6 (chain B): 56.678 Å
Distance between residue 1 (chain A) and residue 7 (chain B): 60.148 Å
Distance between residue 1 (chain A) and residue 8 (chain B): 60.639 Å
Distance between residue 1 (chain A) and residue 9 (chain B): 61.220 Å
Distance between residue 1 (chain A) and residue 10 (chain B): 63.170 Å


In [ ]:
# Calculate minimum all-atom distances for every residue pair.
# Requires: structure from Cell 2 and math imported earlier; set chain IDs.
# For each cross-chain residue pair, the smallest atom-pair distance is kept.
# Output: residue_distances contains (residue_A, residue_B, minimum distance);
# the first 1000 pairs are printed in Angstrom, without contact filtering.
# This overwrites the CA-distance results from Cell 6 and may run slowly.
# Use one model and unique integer residue numbers; all atoms/residues in
# the chosen chains are included, including non-protein residues if present.

def get_all_atoms_coordinates_from_residue(protein_structure, chain_id, residue_id):
    atom_coordinates = []
    for model in protein_structure:
        for chain in model:
            if chain.get_id() == chain_id:  # Filter atoms by chain ID
                for residue in chain:
                    if residue.get_id()[1] == residue_id:  # Filter by residue ID
                        for atom in residue:
                            atom_coordinates.append(atom.get_coord())  # Get atom coordinates
    return atom_coordinates

# Example: Protein 1 is in chain 'A' and Protein 2 is in chain 'B'
chain_id_protein1 = 'A'  # Protein 1 chain
chain_id_protein2 = 'B'  # Protein 2 chain

# Get residue IDs for all residues in both proteins (chains A and B)
protein1_residues = {residue.get_id()[1]: residue for model in structure for chain in model for residue in chain if chain.get_id() == chain_id_protein1}
protein2_residues = {residue.get_id()[1]: residue for model in structure for chain in model for residue in chain if chain.get_id() == chain_id_protein2}

# Function to calculate Euclidean distance between two points
def calculate_distance(atom1, atom2):
    return math.sqrt((atom2[0] - atom1[0])**2 + (atom2[1] - atom1[1])**2 + (atom2[2] - atom1[2])**2)

# Calculate residue-based distance between all residues in protein 1 and all residues in protein 2
residue_distances = []

for res_id_1, residue_1 in protein1_residues.items():
    for res_id_2, residue_2 in protein2_residues.items():
        # Get all atom coordinates for both residues
        atoms_residue1 = get_all_atoms_coordinates_from_residue(structure, chain_id_protein1, res_id_1)
        atoms_residue2 = get_all_atoms_coordinates_from_residue(structure, chain_id_protein2, res_id_2)

        # Initialize the minimum distance as a large value
        min_distance = float('inf')

        # Calculate the distance between every pair of atoms from the two residues
        for atom1 in atoms_residue1:
            for atom2 in atoms_residue2:
                distance = calculate_distance(atom1, atom2)
                min_distance = min(min_distance, distance)  # Keep track of the minimum distance

        # Store the minimum distance for this residue pair
        residue_distances.append((res_id_1, res_id_2, min_distance))

# Print the distances (for simplicity, print the first 10 distances)
for i, (res_id_1, res_id_2, distance) in enumerate(residue_distances[:1000]):
    print(f"Distance between residue {res_id_1} (chain {chain_id_protein1}) and residue {res_id_2} (chain {chain_id_protein2}): {distance:.3f} Å")

Distance between residue 1 (chain A) and residue 1 (chain B): 53.419 Å
Distance between residue 1 (chain A) and residue 2 (chain B): 54.456 Å
Distance between residue 1 (chain A) and residue 3 (chain B): 50.508 Å
Distance between residue 1 (chain A) and residue 4 (chain B): 53.518 Å
Distance between residue 1 (chain A) and residue 5 (chain B): 55.631 Å
Distance between residue 1 (chain A) and residue 6 (chain B): 51.045 Å
Distance between residue 1 (chain A) and residue 7 (chain B): 56.942 Å
Distance between residue 1 (chain A) and residue 8 (chain B): 58.554 Å
Distance between residue 1 (chain A) and residue 9 (chain B): 56.805 Å
Distance between residue 1 (chain A) and residue 10 (chain B): 60.606 Å
Distance between residue 1 (chain A) and residue 11 (chain B): 55.261 Å
Distance between residue 1 (chain A) and residue 12 (chain B): 61.044 Å
Distance between residue 1 (chain A) and residue 13 (chain B): 63.927 Å
Distance between residue 1 (chain A) and residue 14 (chain B): 64.158 Å
D

In [ ]:
def get_all_atoms_coordinates_from_residue(protein_structure, chain_id, residue_id):
    atom_coordinates = []
    for model in protein_structure:
        for chain in model:
            if chain.get_id() == chain_id:  # Filter atoms by chain ID
                for residue in chain:
                    if residue.get_id()[1] == residue_id:  # Filter by residue ID
                        for atom in residue:
                            atom_coordinates.append(atom.get_coord())  # Get atom coordinates
    return atom_coordinates

# Example: Protein 1 is in chain 'A' and Protein 2 is in chain 'B'
chain_id_protein1 = 'A'  # Protein 1 chain
chain_id_protein2 = 'B'  # Protein 2 chain

# Get residue IDs for all residues in both proteins (chains A and B)
protein1_residues = {residue.get_id()[1]: residue for model in structure for chain in model for residue in chain if chain.get_id() == chain_id_protein1}
protein2_residues = {residue.get_id()[1]: residue for model in structure for chain in model for residue in chain if chain.get_id() == chain_id_protein2}

# Function to calculate Euclidean distance between two points
def calculate_distance(atom1, atom2):
    return math.sqrt((atom2[0] - atom1[0])**2 + (atom2[1] - atom1[1])**2 + (atom2[2] - atom1[2])**2)

# Calculate residue-based distance between all residues in protein 1 and all residues in protein 2
residue_distances = []

for res_id_1, residue_1 in protein1_residues.items():
    for res_id_2, residue_2 in protein2_residues.items():
        # Get all atom coordinates for both residues
        atoms_residue1 = get_all_atoms_coordinates_from_residue(structure, chain_id_protein1, res_id_1)
        atoms_residue2 = get_all_atoms_coordinates_from_residue(structure, chain_id_protein2, res_id_2)

        # Initialize the minimum distance as a large value
        min_distance = float('inf')

        # Calculate the distance between every pair of atoms from the two residues
        for atom1 in atoms_residue1:
            for atom2 in atoms_residue2:
                distance = calculate_distance(atom1, atom2)
                min_distance = min(min_distance, distance)  # Keep track of the minimum distance

        # Store the minimum distance for this residue pair
        residue_distances.append((res_id_1, res_id_2, min_distance))
        if min_distance < 3.0:
            print(f"Distance between residue {res_id_1} (chain {chain_id_protein1}) and residue {res_id_2} (chain {chain_id_protein2}): {min_distance:.3f} Å")


Distance between residue 13 (chain A) and residue 288 (chain B): 2.837 Å
Distance between residue 14 (chain A) and residue 288 (chain B): 2.770 Å
Distance between residue 17 (chain A) and residue 287 (chain B): 2.716 Å
Distance between residue 48 (chain A) and residue 324 (chain B): 2.710 Å
Distance between residue 48 (chain A) and residue 325 (chain B): 2.516 Å
Distance between residue 75 (chain A) and residue 325 (chain B): 2.873 Å
Distance between residue 92 (chain A) and residue 267 (chain B): 2.395 Å
Distance between residue 179 (chain A) and residue 277 (chain B): 2.996 Å
Distance between residue 179 (chain A) and residue 278 (chain B): 2.471 Å
Distance between residue 179 (chain A) and residue 279 (chain B): 2.422 Å
Distance between residue 182 (chain A) and residue 319 (chain B): 2.782 Å
Distance between residue 182 (chain A) and residue 332 (chain B): 2.956 Å
Distance between residue 202 (chain A) and residue 317 (chain B): 2.555 Å
Distance between residue 228 (chain A) and re

### Export interface residues
Run after uploading the PDB and selecting chains (A and B by default). Exports one row per contacting residue from either chain, using the first model and a strict minimum atom distance < 3.0 Å. All atoms are included, as in the original calculation; nonstandard residues can appear. This cell does not require the later overlap .dat file.


In [ ]:
# Save and download the interface-residue CSV.
# Requires: structure and pdb_file from Cell 2, plus chain_id_protein1 and
# chain_id_protein2 set by an earlier chain-selection cell (default A/B).
# This cell recomputes contacts; it does not use residue_distances.
# Edit cutoff_angstrom below (default 3.0 A, strict <) independently of Cell 8.
# Only structure[0] is analyzed; all atoms and non-protein residues are included.
# Output: <PDB_filename_stem>_interface_residues.csv, automatically downloaded.
# Each contacting residue appears once, with chain, number, insertion code,
# residue name, hetero flag, and minimum distance to the opposite chain.
# A headers-only CSV means no contacts passed the cutoff.
# This CSV is not the I/J partner-overlap table required by Cells 11-14.

import csv
from pathlib import Path
from Bio.PDB import NeighborSearch
from google.colab import files

cutoff_angstrom = 3.0
model = structure[0]
chain_a_id = chain_id_protein1
chain_b_id = chain_id_protein2
if chain_a_id == chain_b_id:
    raise ValueError("Choose two different chain IDs.")
for chain_id in (chain_a_id, chain_b_id):
    if chain_id not in model:
        raise ValueError(f"Chain {chain_id!r} is missing from the first model.")

atoms_b = list(model[chain_b_id].get_atoms())
if not atoms_b:
    raise ValueError("The second chain contains no atoms.")
neighbor_search = NeighborSearch(atoms_b)
interface = {}

def record_residue(atom, distance):
    residue = atom.get_parent()
    chain_id = residue.get_parent().id
    key = (chain_id, residue.id)
    if key not in interface or distance < interface[key][1]:
        interface[key] = (residue, distance)

for atom_a in model[chain_a_id].get_atoms():
    for atom_b in neighbor_search.search(atom_a.coord, cutoff_angstrom, level="A"):
        distance = float(atom_a - atom_b)
        if distance < cutoff_angstrom:
            record_residue(atom_a, distance)
            record_residue(atom_b, distance)

output_file = f"{Path(pdb_file).stem}_interface_residues.csv"
with open(output_file, "w", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["chain", "residue_number", "insertion_code",
                     "residue_name", "hetero_flag", "min_distance_A"])
    for (chain_id, residue_id), (residue, distance) in sorted(interface.items()):
        writer.writerow([chain_id, residue_id[1], residue_id[2].strip(),
                         residue.resname, residue_id[0].strip(),
                         f"{distance:.4f}"])
print(f"Saved {len(interface)} interface residues to {output_file}")
if not interface:
    print("No contacts satisfy the cutoff; the CSV contains headers only.")
files.download(output_file)


Saved 45 interface residues to fold_fgfr1_dimer_extracellular_model_0_interface_residues.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#  Upload the optional multi-partner interface-overlap table.
# Skip Cells 11-14 if you only need interface residues from one complex.
# Input: FGFR1_InterfaceOverlap (3).dat (whitespace-delimited, headers I J).
# I is the partner ID; J is a contacted residue on the SAME target protein.
# Prepare this table separately from contact analyses of each complex;
# all complexes must use a consistent target-protein residue numbering scheme.
# For example, rows '1 25' and '2 25' record shared target residue 25.
# Output: uploaded table available for reading in Cell 12.

from google.colab import files
uploaded = files.upload()

Saving FGFR1_InterfaceOverlap.dat to FGFR1_InterfaceOverlap (3).dat


In [ ]:
# Cell 12 - Read and inspect the partner-interface table.
# Requires: the table uploaded in Cell 11 and pandas installed.
# Edit the pd.read_csv filename if it differs from FGFR1_InterfaceOverlap (3).dat.
# Required whitespace-delimited headers: I J (partner ID, target residue ID).
# Output: dfX, a preview, and column/type information.
# If parsing fails, correct the table before continuing. The existing error
# handler refers to a DIFFERENT similarity-table filename and does not
# recover dfX; it may itself fail and should not be used as input guidance.

import pandas as pd

# Read the CSV file with error handling and flexible delimiter
try:
    dfX = pd.read_csv(
        'FGFR1_InterfaceOverlap (3).dat',
        # Removed delimiter=' '
        index_col=False,
        # Handle errors by skipping bad lines or raising warnings
        # on_bad_lines='skip',  # Skip lines with too many fields
        # on_bad_lines='warn',  # Print a warning for bad lines
        # Flexible delimiter to handle extra spaces
        delim_whitespace=True,
    )
except pd.errors.ParserError as e:
    print(f"Error reading file: {e}")
    # Inspect line 227 manually for inconsistencies
    with open('PPICellAdhesion_InterCellType_JaccardSimilarity.dat', 'r') as f:
        for i, line in enumerate(f):
            if i == 226:  # Line numbers are 0-based
                print(f"Line 227: {line.strip()}")
                break

# Print some info if successful
if dfX is not None:
    print(dfX.head())
    print(dfX.info())

   I   J
0  1  20
1  1  28
2  1  31
3  1  64
4  1  66
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   I       153 non-null    int64
 1   J       153 non-null    int64
dtypes: int64(2)
memory usage: 2.5 KB
None


<ipython-input-24-2cfe563f403e>:5: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  dfX = pd.read_csv(


In [ ]:
#  Select partner IDs and target-residue IDs for overlap analysis.
# Requires: dfX successfully loaded in Cell 12, with exact headers I and J.
# Output: column_1 (partner IDs) and column_2 (target-residue IDs), printed.
# Both columns must be numeric. Target residue numbers must refer to the
# same protein and numbering scheme across all partners.

# Access columns using their index or actual names
column_1 = dfX['I']  # Changed df to dfX
column_2 = dfX['J']  # Changed df to dfX


# Print the specific columns
print(column_1)
print(column_2)


0      1
1      1
2      1
3      1
4      1
      ..
148    8
149    8
150    8
151    8
152    8
Name: I, Length: 153, dtype: int64
0       20
1       28
2       31
3       64
4       66
      ... 
148    317
149    319
150    324
151    325
152    325
Name: J, Length: 153, dtype: int64


In [ ]:
#  Report whether each pair of partners shares a target residue.
# Requires: column_1 and column_2 from Cell 13.
# Current ranges compare partner IDs 1..8 and target residue IDs 0..349;
# edit ALL relevant range limits below for a different dataset.
# Output: partner_i partner_j flag, where 1 means at least one shared
# residue was found in that range and 0 means none was found there.
# Results are printed only; no overlap file is saved.
# Shared residues suggest interface overlap, but do not prove competitive
# binding; non-overlapping lists also do not establish simultaneous binding.

for i in range(1, 8):
    for j in range(i+1, 9):
        flag_i = 0
        flag_j = 0
        for k in range(0, len(column_1)):  # Change here to use len(column_1)
            # print(i , j , column_1[k], column_2[k])
            for l in range(0,350):
                if column_1[k] == i and column_2[k] == l:
                    flag_i = 1
                    for m in range(0, len(column_1)):  # Change here to use len(column_1)
                        if column_1[m] == j and column_2[m] == l:
                            flag_j = 1

        if flag_i ==1 and flag_j ==1:
            print(i , j , 1) # 1 means overlap
        else:
            print(i , j, 0) # 0 means not overlap

1 2 0
1 3 0
1 4 0
1 5 1
1 6 1
1 7 1
1 8 1
2 3 1
2 4 1
2 5 1
2 6 0
2 7 1
2 8 1
3 4 0
3 5 0
3 6 0
3 7 0
3 8 0
4 5 1
4 6 0
4 7 1
4 8 1
5 6 1
5 7 1
5 8 1
6 7 1
6 8 1
7 8 1
